# CPU patch-path diagnosis and four-task control revalidation
No model, no GPU, no agent performance measurement.

In [ ]:
# ============================ CONFIG ============================
DISPATCH_CONFIRM = False     # guards model startup/dispatch; a Kaggle GPU session still consumes quota during setup
SANDBOX_ROOT_STR = "/tmp"    # where swegemma creates its sandbox directories
CLEANUP_GRACE_S  = 45        # bounded wait for THIS run's sandboxes to be torn down
SESSION_CAP_MIN  = 300       # admission deadline; not a hard OS-process/session kill
RUN_RESERVE_MIN = 25         # refuse a new run when less than this planning allowance remains

# Identical for A and B. Not tuned per variant.
BUDGET = dict(max_time_minutes=10.0, max_tool_calls=100, max_turns=60, timeout_seconds=300)
# ================================================================
import json, os, sys, time
from pathlib import Path
T0 = time.perf_counter()
def elapsed_min():
    return (time.perf_counter() - T0) / 60.0
print("compare config:", BUDGET, "| session cap", SESSION_CAP_MIN, "min")
if not DISPATCH_CONFIRM:
    print("DISPATCH_CONFIRM is False: setup and preconditions will run, the eight runs will NOT.")


In [ ]:
# Wheelhouse install. Exit status is checked, not piped away.
import glob, importlib, os, subprocess, sys
from pathlib import Path
os.environ.update({
    "LITELLM_LOCAL_MODEL_COST_MAP": "True", "TRANSFORMERS_NO_TF": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1",
    "VLLM_ENGINE_READY_TIMEOUT_S": "1200", "VLLM_NO_USAGE_STATS": "1",
    "OTEL_SDK_DISABLED": "true", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
WHEELHOUSE_DIR = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
            "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path("/tmp/wheelhouse"); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in w.name.lower():
        continue
    name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
wheels = sorted(str(w) for w in tmp_whl.glob("*.whl"))
print("installing", len(wheels), "wheels...")
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                       "--force-reinstall", *wheels], capture_output=True, text=True)
if proc.returncode != 0:
    print(proc.stdout[-2000:]); print(proc.stderr[-2000:])
    raise SystemExit(f"wheelhouse install failed rc={proc.returncode}")
importlib.invalidate_caches()
import importlib.metadata as md
VERSIONS = {}
for p in ("swegemma", "adk-submission", "adk-eval-core", "google-adk", "vllm", "transformers", "litellm"):
    try:
        VERSIONS[p] = md.version(p)
    except Exception as e:
        VERSIONS[p] = "MISSING: " + repr(e)[:60]
print(json.dumps(VERSIONS, indent=2))


In [ ]:
import hashlib, json
from pathlib import Path
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
RESULTS = WORKING_DIR / 'pilot'
RESULTS.mkdir(parents=True, exist_ok=True)
assert DATA_DIR.exists(), 'competition data not mounted'
print('CPU-only control preflight: no model attached or started')


In [ ]:
# Shared helpers. Defined BEFORE any cell that uses them (the previous notebook referenced
# run_sync / TASK_IDS from the repair cell, which were only defined later -> NameError at runtime).
import asyncio, concurrent.futures, re
from swegemma.models import load_tasks

TASKS_PATH = DATA_DIR / "tasks.jsonl"
all_tasks = load_tasks(TASKS_PATH)
ALL_BY_ID = {t.instance_id: t for t in all_tasks}
PKG = {"fastapi/fastapi": "fastapi", "Textualize/rich": "rich",
       "psf/requests": "requests", "encode/httpx": "httpx"}

def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def snapshot_for(t):
    return DATA_DIR / "snapshots" / (t.instance_id + ".tgz")

def target_tests(t):
    return re.findall(r"^\+\+\+ b/(\S+)", t.test_patch or "", re.M)

def pkg_of(t):
    return PKG.get(t.repo, t.repo.split("/")[-1])

def under(path, ws):
    from pathlib import PurePosixPath
    import posixpath
    if not path or not ws or not path.startswith('/') or not ws.startswith('/'):
        return False
    actual, root = PurePosixPath(posixpath.normpath(path)), PurePosixPath(posixpath.normpath(ws))
    return actual != root and actual.is_relative_to(root)

print("loaded", len(all_tasks), "tasks")


import time
from pathlib import Path

def sandbox_set():
    '''Sandbox directories currently on disk. Used to attribute cleanup to a specific run.'''
    return {str(p) for p in Path(SANDBOX_ROOT_STR).glob('swegemma_sandbox_*')}


def confirm_cleanup(before, grace_s=None):
    '''Sandboxes THIS run created that are still present after a bounded grace period.

    Only directories absent before the run are counted, so an unrelated leftover cannot mask or
    manufacture a failure. Nothing is ever deleted here: a leak is reported, not tidied away, and an
    unrelated sandbox is never touched. Returns (ok, owned_leftovers).
    '''
    grace_s = CLEANUP_GRACE_S if grace_s is None else grace_s
    deadline = time.time() + grace_s
    owned = sorted(sandbox_set() - before)
    while owned and time.time() < deadline:
        time.sleep(3)
        owned = sorted(sandbox_set() - before)
    return (not owned), owned


In [ ]:
# Task manifest + the saved control evidence these tasks were selected on.
TASK_IDS = ["rich_3278", "rich_3535", "rich_3675", "rich_3942"]
CONTROL_EVIDENCE = {'rich_3278': {'negative': {'pytest_exit': 1, 'workspace_real': '/tmp/swegemma_sandbox_fbc86660-ea1_mbw02i_a/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_fbc86660-ea1_mbw02i_a/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_ansi::test_decode': 'passed', 'test_ansi::test_decode_example': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b[31mFound 4 errors in 2 files (checked 18 source files)\\x1b(B\\x1b[m\\n-Found 4 errors in 2 files (checked 18 source files)]': 'passed', 'test_ansi::test_decode_issue_2688[Hallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(BHallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(JHallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(BHal\\x1b(Jlo-Hallo]': 'passed', 'test_ansi::test_strip_private_escape_sequences[0]': 'failed', 'test_ansi::test_strip_private_escape_sequences[1]': 'failed', 'test_ansi::test_strip_private_escape_sequences[2]': 'failed', 'test_ansi::test_strip_private_escape_sequences[3]': 'failed', 'test_ansi::test_strip_private_escape_sequences[4]': 'failed', 'test_ansi::test_strip_private_escape_sequences[5]': 'failed', 'test_ansi::test_strip_private_escape_sequences[6]': 'failed', 'test_ansi::test_strip_private_escape_sequences[7]': 'failed', 'test_ansi::test_strip_private_escape_sequences[8]': 'failed', 'test_ansi::test_strip_private_escape_sequences[9]': 'failed', 'test_ansi::test_strip_private_escape_sequences[:]': 'failed', 'test_ansi::test_strip_private_escape_sequences[;]': 'failed', 'test_ansi::test_strip_private_escape_sequences[<]': 'failed', 'test_ansi::test_strip_private_escape_sequences[=]': 'failed', 'test_ansi::test_strip_private_escape_sequences[>]': 'failed', 'test_ansi::test_strip_private_escape_sequences[?]': 'failed'}, 'n_cases': 23, 'failed_nodes': ['test_ansi::test_strip_private_escape_sequences[0]', 'test_ansi::test_strip_private_escape_sequences[1]', 'test_ansi::test_strip_private_escape_sequences[2]', 'test_ansi::test_strip_private_escape_sequences[3]', 'test_ansi::test_strip_private_escape_sequences[4]', 'test_ansi::test_strip_private_escape_sequences[5]', 'test_ansi::test_strip_private_escape_sequences[6]', 'test_ansi::test_strip_private_escape_sequences[7]', 'test_ansi::test_strip_private_escape_sequences[8]', 'test_ansi::test_strip_private_escape_sequences[9]', 'test_ansi::test_strip_private_escape_sequences[:]', 'test_ansi::test_strip_private_escape_sequences[;]', 'test_ansi::test_strip_private_escape_sequences[<]', 'test_ansi::test_strip_private_escape_sequences[=]', 'test_ansi::test_strip_private_escape_sequences[>]', 'test_ansi::test_strip_private_escape_sequences[?]'], 'passed_nodes': ['test_ansi::test_decode', 'test_ansi::test_decode_example', 'test_ansi::test_decode_issue_2688[Hallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(BHal\\x1b(Jlo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(BHallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(JHallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b[31mFound 4 errors in 2 files (checked 18 source files)\\x1b(B\\x1b[m\\n-Found 4 errors in 2 files (checked 18 source files)]'], 'failed': ['test_ansi::test_strip_private_escape_sequences[0]', 'test_ansi::test_strip_private_escape_sequences[1]', 'test_ansi::test_strip_private_escape_sequences[2]', 'test_ansi::test_strip_private_escape_sequences[3]', 'test_ansi::test_strip_private_escape_sequences[4]', 'test_ansi::test_strip_private_escape_sequences[5]', 'test_ansi::test_strip_private_escape_sequences[6]', 'test_ansi::test_strip_private_escape_sequences[7]', 'test_ansi::test_strip_private_escape_sequences[8]', 'test_ansi::test_strip_private_escape_sequences[9]', 'test_ansi::test_strip_private_escape_sequences[:]', 'test_ansi::test_strip_private_escape_sequences[;]', 'test_ansi::test_strip_private_escape_sequences[<]', 'test_ansi::test_strip_private_escape_sequences[=]', 'test_ansi::test_strip_private_escape_sequences[>]', 'test_ansi::test_strip_private_escape_sequences[?]'], 'errored': []}, 'positive': {'pytest_exit': 0, 'workspace_real': '/tmp/swegemma_sandbox_2c91183e-840_38s9ivy0/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_2c91183e-840_38s9ivy0/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_ansi::test_decode': 'passed', 'test_ansi::test_decode_example': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b[31mFound 4 errors in 2 files (checked 18 source files)\\x1b(B\\x1b[m\\n-Found 4 errors in 2 files (checked 18 source files)]': 'passed', 'test_ansi::test_decode_issue_2688[Hallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(BHallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(JHallo-Hallo]': 'passed', 'test_ansi::test_decode_issue_2688[\\x1b(BHal\\x1b(Jlo-Hallo]': 'passed', 'test_ansi::test_strip_private_escape_sequences[0]': 'passed', 'test_ansi::test_strip_private_escape_sequences[1]': 'passed', 'test_ansi::test_strip_private_escape_sequences[2]': 'passed', 'test_ansi::test_strip_private_escape_sequences[3]': 'passed', 'test_ansi::test_strip_private_escape_sequences[4]': 'passed', 'test_ansi::test_strip_private_escape_sequences[5]': 'passed', 'test_ansi::test_strip_private_escape_sequences[6]': 'passed', 'test_ansi::test_strip_private_escape_sequences[7]': 'passed', 'test_ansi::test_strip_private_escape_sequences[8]': 'passed', 'test_ansi::test_strip_private_escape_sequences[9]': 'passed', 'test_ansi::test_strip_private_escape_sequences[:]': 'passed', 'test_ansi::test_strip_private_escape_sequences[;]': 'passed', 'test_ansi::test_strip_private_escape_sequences[<]': 'passed', 'test_ansi::test_strip_private_escape_sequences[=]': 'passed', 'test_ansi::test_strip_private_escape_sequences[>]': 'passed', 'test_ansi::test_strip_private_escape_sequences[?]': 'passed'}, 'n_cases': 23, 'failed_nodes': [], 'passed_nodes': ['test_ansi::test_decode', 'test_ansi::test_decode_example', 'test_ansi::test_decode_issue_2688[Hallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(BHal\\x1b(Jlo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(BHallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b(JHallo-Hallo]', 'test_ansi::test_decode_issue_2688[\\x1b[31mFound 4 errors in 2 files (checked 18 source files)\\x1b(B\\x1b[m\\n-Found 4 errors in 2 files (checked 18 source files)]', 'test_ansi::test_strip_private_escape_sequences[0]', 'test_ansi::test_strip_private_escape_sequences[1]', 'test_ansi::test_strip_private_escape_sequences[2]', 'test_ansi::test_strip_private_escape_sequences[3]', 'test_ansi::test_strip_private_escape_sequences[4]', 'test_ansi::test_strip_private_escape_sequences[5]', 'test_ansi::test_strip_private_escape_sequences[6]', 'test_ansi::test_strip_private_escape_sequences[7]', 'test_ansi::test_strip_private_escape_sequences[8]', 'test_ansi::test_strip_private_escape_sequences[9]', 'test_ansi::test_strip_private_escape_sequences[:]', 'test_ansi::test_strip_private_escape_sequences[;]', 'test_ansi::test_strip_private_escape_sequences[<]', 'test_ansi::test_strip_private_escape_sequences[=]', 'test_ansi::test_strip_private_escape_sequences[>]', 'test_ansi::test_strip_private_escape_sequences[?]'], 'failed': [], 'errored': []}}, 'rich_3535': {'negative': {'pytest_exit': 1, 'workspace_real': '/tmp/swegemma_sandbox_b9bb1fae-ea7_j4kmphm5/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_b9bb1fae-ea7_j4kmphm5/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_cells::test_cell_len_long_string': 'passed', 'test_cells::test_cell_len_short_string': 'passed', 'test_cells::test_set_cell_size': 'passed', 'test_cells::test_set_cell_size_infinite': 'passed', 'test_cells::test_chop_cells': 'passed', 'test_cells::test_chop_cells_double_width_boundary': 'passed', 'test_cells::test_chop_cells_mixed_width': 'passed', 'test_cells::test_is_single_cell_widths': 'failed'}, 'n_cases': 8, 'failed_nodes': ['test_cells::test_is_single_cell_widths'], 'passed_nodes': ['test_cells::test_cell_len_long_string', 'test_cells::test_cell_len_short_string', 'test_cells::test_chop_cells', 'test_cells::test_chop_cells_double_width_boundary', 'test_cells::test_chop_cells_mixed_width', 'test_cells::test_set_cell_size', 'test_cells::test_set_cell_size_infinite'], 'failed': ['test_cells::test_is_single_cell_widths'], 'errored': []}, 'positive': {'pytest_exit': 0, 'workspace_real': '/tmp/swegemma_sandbox_613d24b0-112_yqshur97/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_613d24b0-112_yqshur97/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_cells::test_cell_len_long_string': 'passed', 'test_cells::test_cell_len_short_string': 'passed', 'test_cells::test_set_cell_size': 'passed', 'test_cells::test_set_cell_size_infinite': 'passed', 'test_cells::test_chop_cells': 'passed', 'test_cells::test_chop_cells_double_width_boundary': 'passed', 'test_cells::test_chop_cells_mixed_width': 'passed', 'test_cells::test_is_single_cell_widths': 'passed'}, 'n_cases': 8, 'failed_nodes': [], 'passed_nodes': ['test_cells::test_cell_len_long_string', 'test_cells::test_cell_len_short_string', 'test_cells::test_chop_cells', 'test_cells::test_chop_cells_double_width_boundary', 'test_cells::test_chop_cells_mixed_width', 'test_cells::test_is_single_cell_widths', 'test_cells::test_set_cell_size', 'test_cells::test_set_cell_size_infinite'], 'failed': [], 'errored': []}}, 'rich_3675': {'negative': {'pytest_exit': 1, 'workspace_real': '/tmp/swegemma_sandbox_eee552f1-f25_sg8paafw/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_eee552f1-f25_sg8paafw/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_console::test_dumb_terminal': 'passed', 'test_console::test_soft_wrap': 'passed', 'test_console::test_16color_terminal': 'passed', 'test_console::test_truecolor_terminal': 'passed', 'test_console::test_kitty_terminal': 'passed', 'test_console::test_console_options_update': 'passed', 'test_console::test_console_options_update_height': 'passed', 'test_console::test_init': 'passed', 'test_console::test_size': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[True-no_descriptor_size0-ValueError-ValueError-ValueError-expected_size0]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-no_descriptor_size1-ValueError-ValueError-ValueError-expected_size1]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-stdin_size2-ValueError-ValueError-expected_size2]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-stdout_size3-ValueError-expected_size3]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-ValueError-stderr_size4-expected_size4]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-ValueError-ValueError-expected_size5]': 'passed', 'test_console::test_repr': 'passed', 'test_console::test_print': 'passed', 'test_console::test_print_multiple': 'passed', 'test_console::test_print_text': 'passed', 'test_console::test_print_text_multiple': 'passed', 'test_console::test_print_json': 'passed', 'test_console::test_print_json_error': 'passed', 'test_console::test_print_json_data': 'passed', 'test_console::test_print_json_ensure_ascii': 'passed', 'test_console::test_print_json_with_default_ensure_ascii': 'passed', 'test_console::test_print_json_indent_none': 'passed', 'test_console::test_console_null_file': 'passed', 'test_console::test_log': 'passed', 'test_console::test_log_milliseconds': 'passed', 'test_console::test_print_empty': 'passed', 'test_console::test_markup_highlight': 'passed', 'test_console::test_print_style': 'passed', 'test_console::test_show_cursor': 'passed', 'test_console::test_clear': 'passed', 'test_console::test_clear_no_terminal': 'passed', 'test_console::test_get_style': 'passed', 'test_console::test_get_style_default': 'passed', 'test_console::test_get_style_error': 'passed', 'test_console::test_render_error': 'passed', 'test_console::test_control': 'passed', 'test_console::test_capture': 'passed', 'test_console::test_input': 'passed', 'test_console::test_input_password': 'passed', 'test_console::test_status': 'passed', 'test_console::test_justify_none': 'passed', 'test_console::test_justify_left': 'passed', 'test_console::test_justify_center': 'passed', 'test_console::test_justify_right': 'passed', 'test_console::test_justify_renderable_none': 'passed', 'test_console::test_justify_renderable_left': 'passed', 'test_console::test_justify_renderable_center': 'passed', 'test_console::test_justify_renderable_right': 'passed', 'test_console::test_render_broken_renderable': 'passed', 'test_console::test_export_text': 'passed', 'test_console::test_export_html': 'passed', 'test_console::test_export_html_inline': 'passed', 'test_console::test_export_svg': 'passed', 'test_console::test_export_svg_specified_unique_id': 'passed', 'test_console::test_save_svg': 'passed', 'test_console::test_save_text': 'passed', 'test_console::test_save_html': 'passed', 'test_console::test_no_wrap': 'passed', 'test_console::test_unicode_error': 'passed', 'test_console::test_bell': 'passed', 'test_console::test_pager': 'passed', 'test_console::test_out': 'passed', 'test_console::test_render_group': 'passed', 'test_console::test_render_group_fit': 'passed', 'test_console::test_get_time': 'passed', 'test_console::test_console_style': 'passed', 'test_console::test_no_color': 'passed', 'test_console::test_quiet': 'passed', 'test_console::test_no_nested_live': 'passed', 'test_console::test_screen': 'passed', 'test_console::test_screen_update': 'passed', 'test_console::test_height': 'passed', 'test_console::test_columns_env': 'passed', 'test_console::test_lines_env': 'passed', 'test_console::test_screen_update_class': 'passed', 'test_console::test_is_alt_screen': 'passed', 'test_console::test_set_console_title': 'passed', 'test_console::test_update_screen': 'passed', 'test_console::test_update_screen_lines': 'passed', 'test_console::test_update_options_markup': 'passed', 'test_console::test_print_width_zero': 'passed', 'test_console::test_size_properties': 'passed', 'test_console::test_print_newline_start': 'passed', 'test_console::test_is_terminal_broken_file': 'passed', 'test_console::test_detect_color_system': 'passed', 'test_console::test_reset_height': 'passed', 'test_console::test_render_lines_height_minus_vertical_pad_is_negative': 'passed', 'test_console::test_recording_no_stdout_and_no_stderr_files': 'passed', 'test_console::test_capturing_no_stdout_and_no_stderr_files': 'passed', 'test_console::test_force_color': 'passed', 'test_console::test_force_color_jupyter': 'passed', 'test_console::test_reenable_highlighting': 'passed', 'test_console::test_brokenpipeerror': 'passed', 'test_console::test_capture_and_record': 'passed', 'test_console::test_tty_compatible': 'failed'}, 'n_cases': 99, 'failed_nodes': ['test_console::test_tty_compatible'], 'passed_nodes': ['test_console::test_16color_terminal', 'test_console::test_bell', 'test_console::test_brokenpipeerror', 'test_console::test_capture', 'test_console::test_capture_and_record', 'test_console::test_capturing_no_stdout_and_no_stderr_files', 'test_console::test_clear', 'test_console::test_clear_no_terminal', 'test_console::test_columns_env', 'test_console::test_console_null_file', 'test_console::test_console_options_update', 'test_console::test_console_options_update_height', 'test_console::test_console_style', 'test_console::test_control', 'test_console::test_detect_color_system', 'test_console::test_dumb_terminal', 'test_console::test_export_html', 'test_console::test_export_html_inline', 'test_console::test_export_svg', 'test_console::test_export_svg_specified_unique_id', 'test_console::test_export_text', 'test_console::test_force_color', 'test_console::test_force_color_jupyter', 'test_console::test_get_style', 'test_console::test_get_style_default', 'test_console::test_get_style_error', 'test_console::test_get_time', 'test_console::test_height', 'test_console::test_init', 'test_console::test_input', 'test_console::test_input_password', 'test_console::test_is_alt_screen', 'test_console::test_is_terminal_broken_file', 'test_console::test_justify_center', 'test_console::test_justify_left', 'test_console::test_justify_none', 'test_console::test_justify_renderable_center', 'test_console::test_justify_renderable_left', 'test_console::test_justify_renderable_none', 'test_console::test_justify_renderable_right', 'test_console::test_justify_right', 'test_console::test_kitty_terminal', 'test_console::test_lines_env', 'test_console::test_log', 'test_console::test_log_milliseconds', 'test_console::test_markup_highlight', 'test_console::test_no_color', 'test_console::test_no_nested_live', 'test_console::test_no_wrap', 'test_console::test_out', 'test_console::test_pager', 'test_console::test_print', 'test_console::test_print_empty', 'test_console::test_print_json', 'test_console::test_print_json_data', 'test_console::test_print_json_ensure_ascii', 'test_console::test_print_json_error', 'test_console::test_print_json_indent_none', 'test_console::test_print_json_with_default_ensure_ascii', 'test_console::test_print_multiple', 'test_console::test_print_newline_start', 'test_console::test_print_style', 'test_console::test_print_text', 'test_console::test_print_text_multiple', 'test_console::test_print_width_zero', 'test_console::test_quiet', 'test_console::test_recording_no_stdout_and_no_stderr_files', 'test_console::test_reenable_highlighting', 'test_console::test_render_broken_renderable', 'test_console::test_render_error', 'test_console::test_render_group', 'test_console::test_render_group_fit', 'test_console::test_render_lines_height_minus_vertical_pad_is_negative', 'test_console::test_repr', 'test_console::test_reset_height', 'test_console::test_save_html', 'test_console::test_save_svg', 'test_console::test_save_text', 'test_console::test_screen', 'test_console::test_screen_update', 'test_console::test_screen_update_class', 'test_console::test_set_console_title', 'test_console::test_show_cursor', 'test_console::test_size', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-ValueError-ValueError-expected_size5]', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-ValueError-stderr_size4-expected_size4]', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-stdout_size3-ValueError-expected_size3]', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-stdin_size2-ValueError-ValueError-expected_size2]', 'test_console::test_size_can_fall_back_to_std_descriptors[False-no_descriptor_size1-ValueError-ValueError-ValueError-expected_size1]', 'test_console::test_size_can_fall_back_to_std_descriptors[True-no_descriptor_size0-ValueError-ValueError-ValueError-expected_size0]', 'test_console::test_size_properties', 'test_console::test_soft_wrap', 'test_console::test_status', 'test_console::test_truecolor_terminal', 'test_console::test_unicode_error', 'test_console::test_update_options_markup', 'test_console::test_update_screen', 'test_console::test_update_screen_lines'], 'failed': ['test_console::test_tty_compatible'], 'errored': []}, 'positive': {'pytest_exit': 0, 'workspace_real': '/tmp/swegemma_sandbox_29800dba-55b_9nqi_yv4/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_29800dba-55b_9nqi_yv4/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_console::test_dumb_terminal': 'passed', 'test_console::test_soft_wrap': 'passed', 'test_console::test_16color_terminal': 'passed', 'test_console::test_truecolor_terminal': 'passed', 'test_console::test_kitty_terminal': 'passed', 'test_console::test_console_options_update': 'passed', 'test_console::test_console_options_update_height': 'passed', 'test_console::test_init': 'passed', 'test_console::test_size': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[True-no_descriptor_size0-ValueError-ValueError-ValueError-expected_size0]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-no_descriptor_size1-ValueError-ValueError-ValueError-expected_size1]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-stdin_size2-ValueError-ValueError-expected_size2]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-stdout_size3-ValueError-expected_size3]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-ValueError-stderr_size4-expected_size4]': 'passed', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-ValueError-ValueError-expected_size5]': 'passed', 'test_console::test_repr': 'passed', 'test_console::test_print': 'passed', 'test_console::test_print_multiple': 'passed', 'test_console::test_print_text': 'passed', 'test_console::test_print_text_multiple': 'passed', 'test_console::test_print_json': 'passed', 'test_console::test_print_json_error': 'passed', 'test_console::test_print_json_data': 'passed', 'test_console::test_print_json_ensure_ascii': 'passed', 'test_console::test_print_json_with_default_ensure_ascii': 'passed', 'test_console::test_print_json_indent_none': 'passed', 'test_console::test_console_null_file': 'passed', 'test_console::test_log': 'passed', 'test_console::test_log_milliseconds': 'passed', 'test_console::test_print_empty': 'passed', 'test_console::test_markup_highlight': 'passed', 'test_console::test_print_style': 'passed', 'test_console::test_show_cursor': 'passed', 'test_console::test_clear': 'passed', 'test_console::test_clear_no_terminal': 'passed', 'test_console::test_get_style': 'passed', 'test_console::test_get_style_default': 'passed', 'test_console::test_get_style_error': 'passed', 'test_console::test_render_error': 'passed', 'test_console::test_control': 'passed', 'test_console::test_capture': 'passed', 'test_console::test_input': 'passed', 'test_console::test_input_password': 'passed', 'test_console::test_status': 'passed', 'test_console::test_justify_none': 'passed', 'test_console::test_justify_left': 'passed', 'test_console::test_justify_center': 'passed', 'test_console::test_justify_right': 'passed', 'test_console::test_justify_renderable_none': 'passed', 'test_console::test_justify_renderable_left': 'passed', 'test_console::test_justify_renderable_center': 'passed', 'test_console::test_justify_renderable_right': 'passed', 'test_console::test_render_broken_renderable': 'passed', 'test_console::test_export_text': 'passed', 'test_console::test_export_html': 'passed', 'test_console::test_export_html_inline': 'passed', 'test_console::test_export_svg': 'passed', 'test_console::test_export_svg_specified_unique_id': 'passed', 'test_console::test_save_svg': 'passed', 'test_console::test_save_text': 'passed', 'test_console::test_save_html': 'passed', 'test_console::test_no_wrap': 'passed', 'test_console::test_unicode_error': 'passed', 'test_console::test_bell': 'passed', 'test_console::test_pager': 'passed', 'test_console::test_out': 'passed', 'test_console::test_render_group': 'passed', 'test_console::test_render_group_fit': 'passed', 'test_console::test_get_time': 'passed', 'test_console::test_console_style': 'passed', 'test_console::test_no_color': 'passed', 'test_console::test_quiet': 'passed', 'test_console::test_no_nested_live': 'passed', 'test_console::test_screen': 'passed', 'test_console::test_screen_update': 'passed', 'test_console::test_height': 'passed', 'test_console::test_columns_env': 'passed', 'test_console::test_lines_env': 'passed', 'test_console::test_screen_update_class': 'passed', 'test_console::test_is_alt_screen': 'passed', 'test_console::test_set_console_title': 'passed', 'test_console::test_update_screen': 'passed', 'test_console::test_update_screen_lines': 'passed', 'test_console::test_update_options_markup': 'passed', 'test_console::test_print_width_zero': 'passed', 'test_console::test_size_properties': 'passed', 'test_console::test_print_newline_start': 'passed', 'test_console::test_is_terminal_broken_file': 'passed', 'test_console::test_detect_color_system': 'passed', 'test_console::test_reset_height': 'passed', 'test_console::test_render_lines_height_minus_vertical_pad_is_negative': 'passed', 'test_console::test_recording_no_stdout_and_no_stderr_files': 'passed', 'test_console::test_capturing_no_stdout_and_no_stderr_files': 'passed', 'test_console::test_force_color': 'passed', 'test_console::test_force_color_jupyter': 'passed', 'test_console::test_reenable_highlighting': 'passed', 'test_console::test_brokenpipeerror': 'passed', 'test_console::test_capture_and_record': 'passed', 'test_console::test_tty_compatible': 'passed'}, 'n_cases': 99, 'failed_nodes': [], 'passed_nodes': ['test_console::test_16color_terminal', 'test_console::test_bell', 'test_console::test_brokenpipeerror', 'test_console::test_capture', 'test_console::test_capture_and_record', 'test_console::test_capturing_no_stdout_and_no_stderr_files', 'test_console::test_clear', 'test_console::test_clear_no_terminal', 'test_console::test_columns_env', 'test_console::test_console_null_file', 'test_console::test_console_options_update', 'test_console::test_console_options_update_height', 'test_console::test_console_style', 'test_console::test_control', 'test_console::test_detect_color_system', 'test_console::test_dumb_terminal', 'test_console::test_export_html', 'test_console::test_export_html_inline', 'test_console::test_export_svg', 'test_console::test_export_svg_specified_unique_id', 'test_console::test_export_text', 'test_console::test_force_color', 'test_console::test_force_color_jupyter', 'test_console::test_get_style', 'test_console::test_get_style_default', 'test_console::test_get_style_error', 'test_console::test_get_time', 'test_console::test_height', 'test_console::test_init', 'test_console::test_input', 'test_console::test_input_password', 'test_console::test_is_alt_screen', 'test_console::test_is_terminal_broken_file', 'test_console::test_justify_center', 'test_console::test_justify_left', 'test_console::test_justify_none', 'test_console::test_justify_renderable_center', 'test_console::test_justify_renderable_left', 'test_console::test_justify_renderable_none', 'test_console::test_justify_renderable_right', 'test_console::test_justify_right', 'test_console::test_kitty_terminal', 'test_console::test_lines_env', 'test_console::test_log', 'test_console::test_log_milliseconds', 'test_console::test_markup_highlight', 'test_console::test_no_color', 'test_console::test_no_nested_live', 'test_console::test_no_wrap', 'test_console::test_out', 'test_console::test_pager', 'test_console::test_print', 'test_console::test_print_empty', 'test_console::test_print_json', 'test_console::test_print_json_data', 'test_console::test_print_json_ensure_ascii', 'test_console::test_print_json_error', 'test_console::test_print_json_indent_none', 'test_console::test_print_json_with_default_ensure_ascii', 'test_console::test_print_multiple', 'test_console::test_print_newline_start', 'test_console::test_print_style', 'test_console::test_print_text', 'test_console::test_print_text_multiple', 'test_console::test_print_width_zero', 'test_console::test_quiet', 'test_console::test_recording_no_stdout_and_no_stderr_files', 'test_console::test_reenable_highlighting', 'test_console::test_render_broken_renderable', 'test_console::test_render_error', 'test_console::test_render_group', 'test_console::test_render_group_fit', 'test_console::test_render_lines_height_minus_vertical_pad_is_negative', 'test_console::test_repr', 'test_console::test_reset_height', 'test_console::test_save_html', 'test_console::test_save_svg', 'test_console::test_save_text', 'test_console::test_screen', 'test_console::test_screen_update', 'test_console::test_screen_update_class', 'test_console::test_set_console_title', 'test_console::test_show_cursor', 'test_console::test_size', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-ValueError-ValueError-expected_size5]', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-ValueError-stderr_size4-expected_size4]', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-ValueError-stdout_size3-ValueError-expected_size3]', 'test_console::test_size_can_fall_back_to_std_descriptors[False-ValueError-stdin_size2-ValueError-ValueError-expected_size2]', 'test_console::test_size_can_fall_back_to_std_descriptors[False-no_descriptor_size1-ValueError-ValueError-ValueError-expected_size1]', 'test_console::test_size_can_fall_back_to_std_descriptors[True-no_descriptor_size0-ValueError-ValueError-ValueError-expected_size0]', 'test_console::test_size_properties', 'test_console::test_soft_wrap', 'test_console::test_status', 'test_console::test_truecolor_terminal', 'test_console::test_tty_compatible', 'test_console::test_unicode_error', 'test_console::test_update_options_markup', 'test_console::test_update_screen', 'test_console::test_update_screen_lines'], 'failed': [], 'errored': []}}, 'rich_3942': {'negative': {'pytest_exit': 1, 'workspace_real': '/tmp/swegemma_sandbox_cb7391fe-ddf_sauv1peb/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_cb7391fe-ddf_sauv1peb/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_markdown::test_markdown_render': 'failed', 'test_markdown::test_inline_code': 'passed', 'test_markdown::test_markdown_table': 'failed', 'test_markdown::test_inline_styles_in_table': 'failed', 'test_markdown::test_inline_styles_with_justification': 'failed', 'test_markdown::test_partial_table': 'failed', 'test_markdown::test_table_with_empty_cells': 'passed', 'test_markdown_no_hyperlinks::test_markdown_render': 'failed'}, 'n_cases': 8, 'failed_nodes': ['test_markdown::test_inline_styles_in_table', 'test_markdown::test_inline_styles_with_justification', 'test_markdown::test_markdown_render', 'test_markdown::test_markdown_table', 'test_markdown::test_partial_table', 'test_markdown_no_hyperlinks::test_markdown_render'], 'passed_nodes': ['test_markdown::test_inline_code', 'test_markdown::test_table_with_empty_cells'], 'failed': ['test_markdown::test_inline_styles_in_table', 'test_markdown::test_inline_styles_with_justification', 'test_markdown::test_markdown_render', 'test_markdown::test_markdown_table', 'test_markdown::test_partial_table', 'test_markdown_no_hyperlinks::test_markdown_render'], 'errored': []}, 'positive': {'pytest_exit': 0, 'workspace_real': '/tmp/swegemma_sandbox_58f2d1f1-b18_zzqdtsd6/workspace', 'agent_pkg_file': 'unavailable (no agent ran in the control)', 'pytest_pkg_file': '/tmp/swegemma_sandbox_58f2d1f1-b18_zzqdtsd6/workspace/rich/__init__.py', 'import_in_checkout': True, 'patch_rc': 0, 'test_patch_rc': 0, 'nodes': {'test_markdown::test_markdown_render': 'passed', 'test_markdown::test_inline_code': 'passed', 'test_markdown::test_markdown_table': 'passed', 'test_markdown::test_inline_styles_in_table': 'passed', 'test_markdown::test_inline_styles_with_justification': 'passed', 'test_markdown::test_partial_table': 'passed', 'test_markdown::test_table_with_empty_cells': 'passed', 'test_markdown_no_hyperlinks::test_markdown_render': 'passed'}, 'n_cases': 8, 'failed_nodes': [], 'passed_nodes': ['test_markdown::test_inline_code', 'test_markdown::test_inline_styles_in_table', 'test_markdown::test_inline_styles_with_justification', 'test_markdown::test_markdown_render', 'test_markdown::test_markdown_table', 'test_markdown::test_partial_table', 'test_markdown::test_table_with_empty_cells', 'test_markdown_no_hyperlinks::test_markdown_render'], 'failed': [], 'errored': []}}}
EXPECTED_TASK_HASHES = {"rich_3278": "8fb8127fdb3e069ae02db8a0e3d2e5306b96e7d42a1ab7c3185da2e269cd61bf", "rich_3535": "3c70ac67cc8e78d329459615dd6db711e2715b92f6da4632cc4905f1faf0ad18", "rich_3675": "d651989d2710c16333584401276a29f5c36026b5fff3407204f53d550f3176fb", "rich_3942": "b626009683a3a55af3ad72e8a4cc9cd629cb24444a2c404c073a87778b8148b8"}
import hashlib
def task_hash(t):
    return hashlib.sha256(json.dumps({k:getattr(t,k,None) for k in
        ('repo','base_commit','problem_statement','hints_text','patch','test_patch')},sort_keys=True).encode()).hexdigest()

missing = [t for t in TASK_IDS if t not in ALL_BY_ID]
assert not missing, f"unknown task ids: {missing}"
SELECTED = [ALL_BY_ID[t] for t in TASK_IDS]
for t in SELECTED:
    assert task_hash(t) == EXPECTED_TASK_HASHES.get(t.instance_id), f'{t.instance_id}: task content changed since local preparation; rerun controls'

print("PILOT TASKS and the evidence they were chosen on:")
for tid in TASK_IDS:
    ev = CONTROL_EVIDENCE.get(tid)
    print("\n ", tid, ALL_BY_ID[tid].repo)
    if not ev:
        raise AssertionError(f"{tid}: missing saved control evidence")
    for phase in ("negative", "positive"):
        e = ev.get(phase, {})
        print(f"    {phase:9s} exit={e.get('pytest_exit')} cases={e.get('n_cases')} "
              f"failed={len(e.get('failed', []))} errored={len(e.get('errored', []))}")
        print(f"              agent  import: {e.get('agent_pkg_file','')}")
        print(f"              pytest import: {e.get('pytest_pkg_file','')}")
    neg, pos = ev.get("negative", {}), ev.get("positive", {})
    failed = set(neg.get('failed_nodes', []))
    passed = set(pos.get('passed_nodes', []))
    ok = (neg.get('pytest_exit') == 1 and pos.get('pytest_exit') == 0
          and bool(failed) and failed <= passed
          and not neg.get('errored') and not pos.get('errored')
          and all(under(e.get(key, ''), e.get('workspace_real', ''))
                  for e in (neg,pos) for key in ('pytest_pkg_file',)))
    print(f"    baseline-fails / gold-passes / grading-import-in-checkout: {ok}")
    print("    agent-side import provenance: UNAVAILABLE (no agent ran in the control)")
    print(f"    control patch_rc={neg.get('patch_rc')} test_patch_rc={neg.get('test_patch_rc')}")
    assert ok, f"{tid}: saved control evidence does not support using this task"
    if neg.get("failed"):
        print(f"    baseline failures: {neg['failed'][:4]}")


In [ ]:
# The agent prompt must contain no answer-key text beyond what the issue itself carries.
from swegemma.harness.agent_runner import build_agent_prompt
from dataclasses import replace

class _B:
    time_minutes, tool_calls, turns, cost_usd = (BUDGET["max_time_minutes"], BUDGET["max_tool_calls"],
                                                 BUDGET["max_turns"], None)
class _H:
    command_timeout_seconds, max_stdout_chars, max_file_lines, max_file_chars = (
        BUDGET["timeout_seconds"], 5000, 150, 10000)
class _C:
    budget, harness = _B(), _H()
    graph_dir, embeddings_dir = str(DATA_DIR / "graphs"), str(DATA_DIR / "embeddings")
    enable_sandbox_testing = True

def added_lines(diff):
    return [l[1:].strip() for l in (diff or "").splitlines()
            if l.startswith("+") and not l.startswith("+++") and len(l.strip()) > 40]

leaks, benign = [], 0
for t in SELECTED:
    prompt = build_agent_prompt(task=t, config=_C(), workspace_tree="")
    assert prompt == build_agent_prompt(task=replace(t, patch="SENTINEL", test_patch="SENTINEL"),
                                        config=_C(), workspace_tree=""), "answer keys affect the prompt"
    issue = (t.problem_statement or "") + "\n" + (getattr(t, "hints_text", "") or "")
    for src in ("patch", "test_patch"):
        for l in added_lines(getattr(t, src, "")):
            if l in prompt:
                benign += 1 if l in issue else 0
                if l not in issue:
                    leaks.append((t.instance_id, src, l[:70]))
print("prompt leakage:", "CLEAN" if not leaks else leaks[:3], f"({benign} lines came from the issue text)")
assert not leaks
print("LIMITATION: this is a data-flow check. The subprocess backend runs run_command on the host, so")
print("answer-key files remain reachable; traces must be audited for contamination before trusting a run.")


In [ ]:
# ENVIRONMENT REPAIR (evaluation support only - never part of a submission artifact).
#
# swegemma/harness/container_setup.py returns immediately for the subprocess backend in
# install_editable_package (line 512) and install_test_dependencies (line 529). The sandbox venv
# carries _host_env.pth exposing the Kaggle host's dist-packages, which hold RELEASED
# fastapi/rich/requests/httpx, so without this repair the checkout is never imported.
# The real grader runs --sandbox docker where those functions DO execute. This moves the proxy toward
# Docker; it does NOT make it identical (dependency provisioning still differs).
import swegemma.harness.agent_runner as _ar
import swegemma.harness.verification as _vf
import swegemma.harness.container_setup as _cs

REQUIRED_BACKENDS = ["setuptools", "wheel", "editables"]
OPTIONAL_BACKENDS = ["flit_core", "hatchling", "poetry_core", "pdm_backend"]
COMP_WHEELS = str(DATA_DIR / "wheels")
_SUBPROC = {"SubprocessManager", "SubprocessSandbox"}
REPAIR_LOG = []
ACTIVE_RUN = None
ACTIVE_PHASE = "precondition"

# Idempotent: unwrap any previous patch before wrapping, so re-running this cell cannot nest wrappers.
_ORIG_INSTALL = getattr(_cs.install_editable_package, "__wrapped_original__", None) or _cs.install_editable_package


def _pip(docker, cid, pkg):
    """Run pip and KEEP the exit status; no `| tail` swallowing failures."""
    r = docker.exec(cid, "python3 -m pip install --no-index --find-links=/wheels "
                         "--find-links=" + COMP_WHEELS + " --no-build-isolation " + pkg)
    return {"rc": r.exit_code, "out": (r.stdout or ""), "err": (r.stderr or "")}


def patched_install_editable_package(docker, container_id):
    _ORIG_INSTALL(docker, container_id)
    if type(docker).__name__ not in _SUBPROC:
        return
    entry = {"container": str(container_id), "required": {}, "optional": {}, "phase": ACTIVE_PHASE, "run": ACTIVE_RUN}
    for b in REQUIRED_BACKENDS:
        entry["required"][b] = _pip(docker, container_id, b)
    for b in OPTIONAL_BACKENDS:
        entry["optional"][b] = _pip(docker, container_id, b)
    r = docker.exec(container_id, "cd /workspace && python3 -m pip install --no-index "
                                  "--find-links=/wheels --find-links=" + COMP_WHEELS +
                                  " --no-build-isolation --no-deps -e .")
    entry["editable"] = {"rc": r.exit_code, "out": (r.stdout or ""), "err": (r.stderr or "")}
    REPAIR_LOG.append(entry)
    (RESULTS/"setup_observations.json").write_text(json.dumps(REPAIR_LOG,indent=2),encoding="utf-8")
    failed_required = [b for b, v in entry["required"].items() if v["rc"] != 0]
    if failed_required:
        raise RuntimeError(f"required backend install failed: {failed_required}")
    if entry["editable"]["rc"] != 0:
        raise RuntimeError("editable install of /workspace failed rc="
                           + str(entry["editable"]["rc"]) + " :: " + entry["editable"]["err"][-300:])

    # Check imports inside the actual setup path in each agent/grading sandbox.
    pkg = (ACTIVE_RUN or {}).get('package')
    if not pkg:
        raise RuntimeError('Active task package is unavailable during setup')
    rw = docker.exec(container_id, 'cd /workspace && pwd -P')
    ws = (rw.stdout or '').strip()
    entry['workspace_real'] = ws
    for view, prefix in [('agent', ''), ('grading_flags', 'PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 ')]:
        suffix = '-s ' if view == 'grading_flags' else ''
        cmd = ('cd /workspace && ' + prefix + 'python3 ' + suffix + '-c "import ' + pkg +
               ' as m,sys; from pathlib import Path; print(Path(m.__file__).resolve()); print(sys.executable)"')
        result = docker.exec(container_id, cmd)
        lines = (result.stdout or '').strip().splitlines()
        entry[view] = {'rc': result.exit_code, 'path': lines[0] if lines else '',
                       'executable': lines[1] if len(lines)>1 else '', 'stderr': result.stderr}
    entry['provenance_ok'] = rw.exit_code == 0 and all(
        entry[v]['rc'] == 0 and under(entry[v]['path'], ws) for v in ('agent','grading_flags'))
    # Persist even a failing observation before raising; this is not an in-pytest hook.
    (RESULTS/'setup_observations.json').write_text(json.dumps(REPAIR_LOG,indent=2),encoding='utf-8')
    if not entry['provenance_ok']:
        raise RuntimeError('Actual sandbox import is outside its checkout: ' + json.dumps(entry))


patched_install_editable_package.__wrapped_original__ = _ORIG_INSTALL
for _m in (_ar, _vf, _cs):
    _m.install_editable_package = patched_install_editable_package
assert _ar.install_editable_package is _vf.install_editable_package is _cs.install_editable_package
print("repair bound on agent_runner, verification, container_setup (idempotent)")
print("NOTE: a pip success message alone is not proof; the preconditions below check what actually loads.")


In [ ]:

from swegemma.sandbox import SubprocessManager
_original_exec = SubprocessManager.exec
_audit_path = RESULTS / 'sandbox_commands.jsonl'
def _audited_exec(self, sandbox_id, command, *, timeout=None):
    record = dict(sandbox=str(sandbox_id), command=command)
    try:
        result = _original_exec(self, sandbox_id, command, timeout=timeout)
        record.update(rc=result.exit_code, stdout=result.stdout, stderr=result.stderr)
        return result
    except BaseException as exc:
        record['exception'] = repr(exc)
        raise
    finally:
        with _audit_path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, default=str) + '\n')
SubprocessManager.exec = _audited_exec

ORIGINAL_WRITE_EXPR = '\'python3 -c "import base64,sys,pathlib;pathlib.Path(\\\'/tmp/\' + label + \'.patch\\\').write_bytes(base64.b64decode(sys.argv[1]))" \' + b64'
ORIGINAL_APPLY_EXPR = "'cd /workspace && git apply --verbose /tmp/' + label + '.patch || git apply -3 /tmp/' + label + '.patch'"

# Execute the frozen v1 patch writer and applier with real subprocess commands.
# Separate sandbox: its mutations cannot affect the revalidation arms.
import base64, shlex, traceback
from swegemma.sandbox import SubprocessManager, sandbox_exec, sandbox_start, sandbox_stop
from swegemma.harness.container_setup import extract_snapshot, setup_baseline_commit
from swegemma.config import EvalConfig
from adk_submission import ModelRegistry

async def diagnose_original():
    task = SELECTED[0]
    mgr = SubprocessManager(timeout_seconds=120)
    sid = None
    records = []
    out = RESULTS / 'original_patch_diagnostic.json'
    def save():
        out.write_text(json.dumps(records, indent=2, default=str), encoding='utf-8')
    async def command(label, cmd):
        r = await sandbox_exec(mgr, sid, cmd)
        records.append(dict(label=label, command=cmd, rc=r.exit_code,
                            stdout=r.stdout, stderr=r.stderr))
        save()
        return r
    try:
        sid = await sandbox_start(mgr)
        await asyncio.to_thread(extract_snapshot, mgr, sid, snapshot_for(task))
        await asyncio.to_thread(setup_baseline_commit, mgr, sid, 'diagnostic baseline')
        await command('environment', 'cd /workspace && pwd -P && command -v python3 && git status --short')
        # Exact strings from launched v1, injected by the generator; patch data supplied at runtime.
        diff = task.test_patch
        label = 'testpatch'
        b64 = base64.b64encode(diff.encode('utf-8')).decode('ascii')
        write_cmd = eval(ORIGINAL_WRITE_EXPR, {}, dict(label=label, b64=b64))
        apply_cmd = eval(ORIGINAL_APPLY_EXPR, {}, dict(label=label, b64=b64))
        await command('original_write', write_cmd)
        await command('file_probe', 'ls -l /tmp/testpatch.patch; wc -c /tmp/testpatch.patch; sha256sum /tmp/testpatch.patch')
        await command('original_apply', apply_cmd)
        await command('after_apply', 'cd /workspace && git diff --stat && git status --short')
    except BaseException:
        records.append(dict(exception=traceback.format_exc()))
    finally:
        if sid is not None:
            try:
                await sandbox_stop(mgr, sid)
            except BaseException:
                records.append(dict(teardown_error=traceback.format_exc()))
        save()
    for rec in records:
        print(rec.get('label', 'exception'), rec.get('rc'), str(rec.get('stderr', rec.get('exception', '')))[:500])

run_sync(diagnose_original)

# PRECONDITION for EVERY selected task, using the patched setup path. CPU work only.
from swegemma.sandbox import SubprocessManager, sandbox_exec, sandbox_start, sandbox_stop
from swegemma.harness.container_setup import (
    extract_snapshot, install_test_dependencies, setup_baseline_commit,
    setup_container_wheels, setup_git_exclude, setup_workspace_test_config,
)
from swegemma.config import EvalConfig
import base64 as _b64mod
import hashlib as _hashlib
import tempfile as _tempfile
import re as _re_mod
import os
from swegemma.harness.verification import apply_patch_in_container
import traceback as _tbmod
from adk_submission import ModelRegistry

async def _precondition(task):
    global ACTIVE_RUN, ACTIVE_PHASE
    ACTIVE_RUN = {"task": task.instance_id, "candidate": None, "package": pkg_of(task)}
    ACTIVE_PHASE = "precondition"
    mgr = SubprocessManager(timeout_seconds=600)
    sandboxes_before = sandbox_set()
    sid = await sandbox_start(mgr)
    out = {"task": task.instance_id}
    try:
        cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / "snapshots",
                         results_dir=RESULTS / "precondition", submission_dir=WORKING_DIR,
                         models=ModelRegistry(), sandbox="subprocess", timeout_seconds=600,
                         display_mode="quiet")
        await asyncio.to_thread(setup_container_wheels, mgr, sid, cfg)
        await asyncio.to_thread(extract_snapshot, mgr, sid, snapshot_for(task))
        await asyncio.to_thread(setup_git_exclude, mgr, sid)
        await asyncio.to_thread(_cs.install_editable_package, mgr, sid)   # the patched one
        await asyncio.to_thread(install_test_dependencies, mgr, sid, task.repo, config=cfg)
        await asyncio.to_thread(setup_workspace_test_config, mgr, sid, repo=task.repo)
        await asyncio.to_thread(setup_baseline_commit, mgr, sid, "baseline")
        pkg = pkg_of(task)
        rw = await sandbox_exec(mgr, sid, "cd /workspace && pwd -P")
        out["ws"] = (rw.stdout or "").strip()
        ra = await sandbox_exec(mgr, sid,
                                'cd /workspace && python3 -c "import ' + pkg + ' as m,sys; '
                                'print(m.__file__); print(sys.executable)"')
        lines = (ra.stdout or "").strip().splitlines()
        out["agent_file"] = lines[0] if ra.exit_code == 0 and lines else ""
        out["agent_exe"] = lines[1] if ra.exit_code == 0 and len(lines) > 1 else ""
        rp = await sandbox_exec(mgr, sid, "cd /workspace && PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 "
                                          'python3 -s -c "import ' + pkg + ' as m,sys; '
                                          'print(m.__file__); print(sys.executable)"')
        lines = (rp.stdout or "").strip().splitlines()
        out["grading_file"] = lines[0] if rp.exit_code == 0 and lines else ""
        out["grading_exe"] = lines[1] if rp.exit_code == 0 and len(lines) > 1 else ""
    finally:
        try:
            await sandbox_stop(mgr, sid)
        except BaseException as stop_exc:
            out["teardown_error"] = f"{type(stop_exc).__name__}: {stop_exc}"
        # Teardown returning is not proof of teardown.
        out["cleanup_ok"], out["owned_sandboxes"] = confirm_cleanup(sandboxes_before)
        ACTIVE_RUN = None
    return out

PRECONDITIONS = []
for _t in SELECTED:
    _pc = run_sync(lambda _t=_t: _precondition(_t))
    ok = (under(_pc.get("agent_file", ""), _pc.get("ws", ""))
          and under(_pc.get("grading_file", ""), _pc.get("ws", ""))
          and not _pc.get("teardown_error")
          and _pc.get("cleanup_ok") is True)
    if _pc.get("teardown_error"):
        print("    TEARDOWN ERROR: " + str(_pc.get("teardown_error"))[:150])
    if not _pc.get("cleanup_ok"):
        print("    SANDBOX NOT CLEANED UP: " + str(_pc.get("owned_sandboxes")))
    _pc["ok"] = ok
    PRECONDITIONS.append(_pc)
    print(f"{_pc['task']:16s} ok={ok}")
    print(f"    workspace: {_pc.get('ws','')}")
    print(f"    agent    : {_pc.get('agent_file','') or '<import failed>'}")
    print(f"    grading  : {_pc.get('grading_file','') or '<import failed>'}")
(RESULTS / "preconditions.json").write_text(json.dumps(PRECONDITIONS, indent=2), encoding="utf-8")
bad = [p["task"] for p in PRECONDITIONS if not p["ok"]]
assert not bad, f"preconditions failed for {bad}; do not start the model"
print("ALL PRECONDITIONS PASSED")
print("Caveat: passing here does not guarantee the same resolution inside every later sandbox; the")
print("per-run setup observations verify imports again in the actual sandboxes. These are grading-flags probes, not direct in-pytest observations.")

CONTROL_PYTEST_CMD = {'rich_3278': 'PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 python3 -s -m pytest tests/test_ansi.py', 'rich_3535': 'PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 python3 -s -m pytest tests/test_cells.py', 'rich_3675': 'PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 python3 -s -m pytest tests/test_console.py', 'rich_3942': 'PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 python3 -s -m pytest tests/_card_render.py tests/test_markdown.py tests/test_markdown_no_hyperlinks.py'}
CONTROL_TARGET_NODES = {'rich_3278': ['test_ansi::test_strip_private_escape_sequences[0]', 'test_ansi::test_strip_private_escape_sequences[1]', 'test_ansi::test_strip_private_escape_sequences[2]', 'test_ansi::test_strip_private_escape_sequences[3]', 'test_ansi::test_strip_private_escape_sequences[4]', 'test_ansi::test_strip_private_escape_sequences[5]', 'test_ansi::test_strip_private_escape_sequences[6]', 'test_ansi::test_strip_private_escape_sequences[7]', 'test_ansi::test_strip_private_escape_sequences[8]', 'test_ansi::test_strip_private_escape_sequences[9]', 'test_ansi::test_strip_private_escape_sequences[:]', 'test_ansi::test_strip_private_escape_sequences[;]', 'test_ansi::test_strip_private_escape_sequences[<]', 'test_ansi::test_strip_private_escape_sequences[=]', 'test_ansi::test_strip_private_escape_sequences[>]', 'test_ansi::test_strip_private_escape_sequences[?]'], 'rich_3535': ['test_cells::test_is_single_cell_widths'], 'rich_3675': ['test_console::test_tty_compatible'], 'rich_3942': ['test_markdown::test_inline_styles_in_table', 'test_markdown::test_inline_styles_with_justification', 'test_markdown::test_markdown_render', 'test_markdown::test_markdown_table', 'test_markdown::test_partial_table', 'test_markdown_no_hyperlinks::test_markdown_render']}

# ---------------------------------------------------------------------------------------------
# Re-validate the four selected controls UNDER THIS NOTEBOOK'S SETUP.
#
# The saved controls came from the CPU screening kernel. Their environment_versions and
# setup_commands were recorded, but this session's versions are only observable here, at run time,
# so equivalence cannot be asserted in advance. Rather than assume it, the controls are re-run with
# the same patched setup path used above, and compared against what was recorded. This is CPU work
# and happens before the model server starts.
#
# The exact pytest command recorded by the screen is re-issued, so a difference in outcome cannot be
# blamed on a different invocation.
# ---------------------------------------------------------------------------------------------
import xml.etree.ElementTree as _ET

def _nodes_from_junit(xml_text):
    """Return (nodes, duplicates, parse_error).

    nodes is None when the report is missing, unparseable or contains no testcase elements. That
    stays distinct from an empty dict for the rest of the pipeline: an earlier version collapsed
    both to {} via `nodes or {}`, which let a missing report satisfy a reference arm whose expected
    failure set is empty. Duplicate node identities are reported rather than silently overwritten.
    """
    if not (xml_text or '').strip():
        return None, [], 'empty report'
    try:
        root = _ET.fromstring(xml_text)
    except Exception as exc:
        return None, [], f'unparseable report: {type(exc).__name__}: {exc}'
    out, dupes = {}, []
    seen_any = False
    for tc in root.iter('testcase'):
        seen_any = True
        cls = (tc.get('classname') or '').split('.')[-1]
        name = tc.get('name') or ''
        key = f'{cls}::{name}' if cls else name
        outcome = 'passed'
        if tc.find('failure') is not None:
            outcome = 'failed'
        elif tc.find('error') is not None:
            outcome = 'errored'
        elif tc.find('skipped') is not None:
            outcome = 'skipped'
        if key in out:
            dupes.append(key)
        out[key] = outcome
    if not seen_any:
        return None, [], 'report contains no testcase elements'
    return out, sorted(set(dupes)), None


def compare_control_nodes(observed, expected, targets, arm):
    """Full node/outcome comparison for a reproducibility control.

    Returns (ok, detail). `observed` may be None, which is always a failure. Every expected node
    must be present with the same outcome; every saved target node must fail in the baseline arm and
    pass in the reference arm. Comparing failure sets alone never proved the reference arm actually
    ran, let alone passed, the tests the baseline failed.
    """
    detail = {'missing': [], 'extra': [], 'changed': [], 'skipped': [],
              'targets_wrong': [], 'n_observed': 0, 'n_expected': len(expected)}
    if observed is None:
        detail['reason'] = 'no usable JUnit report'
        return False, detail
    if not observed:
        detail['reason'] = 'no nodes collected'
        return False, detail
    detail['n_observed'] = len(observed)
    detail['missing'] = sorted(set(expected) - set(observed))
    detail['extra'] = sorted(set(observed) - set(expected))
    detail['changed'] = sorted(f'{n}: expected {expected[n]}, observed {observed[n]}'
                               for n in set(expected) & set(observed)
                               if expected[n] != observed[n])
    detail['skipped'] = sorted(n for n, o in observed.items() if o == 'skipped')
    want_target = 'failed' if arm == 'baseline' else 'passed'
    detail['targets_wrong'] = sorted(
        f'{n}: expected {want_target}, observed {observed.get(n, "ABSENT")}'
        for n in targets if observed.get(n) != want_target)
    ok = not (detail['missing'] or detail['extra'] or detail['changed'] or detail['targets_wrong'])
    if ok:
        detail['reason'] = ''
    else:
        detail['reason'] = 'node map does not match the saved control'
    return ok, detail


async def _revalidate_control(task, arm):
    """Run one control arm in a fresh sandbox under this notebook's setup. arm: baseline|reference."""
    global ACTIVE_RUN, ACTIVE_PHASE
    ACTIVE_RUN = {"task": task.instance_id, "candidate": None, "package": pkg_of(task)}
    ACTIVE_PHASE = "control_" + arm
    mgr = SubprocessManager(timeout_seconds=900)
    # Snapshot BEFORE this arm creates anything, so only its own sandboxes are attributable to it.
    sandboxes_before = sandbox_set()
    sid = await sandbox_start(mgr)
    rec = {"task": task.instance_id, "arm": arm}
    try:
        cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / "snapshots",
                         results_dir=RESULTS / "control", submission_dir=WORKING_DIR,
                         models=ModelRegistry(), sandbox="subprocess", timeout_seconds=900,
                         display_mode="quiet")
        await asyncio.to_thread(setup_container_wheels, mgr, sid, cfg)
        await asyncio.to_thread(extract_snapshot, mgr, sid, snapshot_for(task))
        await asyncio.to_thread(setup_git_exclude, mgr, sid)
        await asyncio.to_thread(_cs.install_editable_package, mgr, sid)
        await asyncio.to_thread(install_test_dependencies, mgr, sid, task.repo, config=cfg)
        await asyncio.to_thread(setup_workspace_test_config, mgr, sid, repo=task.repo)
        await asyncio.to_thread(setup_baseline_commit, mgr, sid, "baseline")

        # PATCH APPLICATION.
        #
        # Comparison version 1 aborted here: all eight test-patch and all four reference-patch
        # applications returned 128. Measured with real git: 128 means the patch file was MISSING
        # or EMPTY ("can't open patch" / "No valid patches in input"); a patch that merely fails to
        # apply returns 1. The old helper wrote the patch with a shell `python3 -c` one-liner and
        # never checked that command's exit code, so a failed write was invisible and every pytest
        # run then passed against an unpatched checkout.
        #
        # This now does exactly what the screening run did for 38 tasks without a single failure:
        # write to a HOST file, copy it in with mgr.copy_to, and apply with the harness's own
        # multi-strategy apply_patch_in_container. Creation is verified before application, and the
        # command output is preserved either way.
        async def _put(text, name):
            with _tempfile.NamedTemporaryFile("w", suffix="_" + name, delete=False,
                                              encoding="utf-8") as f:
                f.write(text if text.endswith("\n") else text + "\n")
                host = Path(f.name)
            try:
                await asyncio.to_thread(mgr.copy_to, sid, host, "/tmp/")
                return "/tmp/" + host.name, len(text.encode("utf-8")), \
                    _hashlib.sha256(text.encode("utf-8")).hexdigest()
            finally:
                os.unlink(host)

        async def _apply(diff, label):
            """Return (rc, evidence). rc None means no patch was requested for this arm."""
            ev = {"label": label, "requested": bool((diff or "").strip())}
            if not ev["requested"]:
                ev["note"] = "no patch requested for this arm (not a successful application)"
                return None, ev
            path, size, sha = await _put(diff, label + ".patch")
            ev.update({"sandbox_path": path, "bytes": size, "sha256": sha})
            # Verify creation BEFORE attempting application, so a missing or empty file is named
            # as such instead of surfacing later as an opaque git exit 128.
            probe = await sandbox_exec(mgr, sid, "wc -c < " + path + " || echo MISSING")
            ev["created_bytes"] = (probe.stdout or "").strip()
            ev["create_probe_rc"] = probe.exit_code
            if probe.exit_code != 0 or not (ev["created_bytes"] or "").strip().isdigit() \
                    or int(ev["created_bytes"]) == 0:
                ev["error"] = "patch file was not created, or is empty; not attempting application"
                return 128, ev
            code, out, err = await asyncio.to_thread(apply_patch_in_container, mgr, sid, path)
            ev.update({"apply_rc": code, "stdout": (out or "")[-1500:], "stderr": (err or "")[-1500:]})
            return code, ev

        # Mirror the screening run: restore any test path before re-applying the verification patch.
        for _tf in _re_mod.findall(r"^\+\+\+ b/(\S+)", getattr(task, "test_patch", "") or "", _re_mod.M):
            await sandbox_exec(mgr, sid, "cd /workspace && git checkout HEAD -- " + _tf)
            await sandbox_exec(mgr, sid, "cd /workspace && git clean -f -- " + _tf)

        rec["test_patch_rc"], rec["test_patch_evidence"] = await _apply(
            getattr(task, "test_patch", ""), "testpatch")
        if arm == "reference":
            rec["patch_rc"], rec["patch_evidence"] = await _apply(getattr(task, "patch", ""), "gold")
        else:
            # NOT 0. The baseline arm applies no source patch; reporting 0 made "nothing was done"
            # indistinguishable from "applied successfully", which is how version 1 read as clean.
            rec["patch_rc"], rec["patch_evidence"] = None, {
                "label": "gold", "requested": False,
                "note": "baseline arm applies no source patch by design"}

        cmd = CONTROL_PYTEST_CMD[task.instance_id]
        xml = "/tmp/ctl_" + arm + ".xml"
        r = await sandbox_exec(mgr, sid, "cd /workspace && " + cmd + " -q --junit-xml=" + xml)
        rec["pytest_exit"] = r.exit_code
        rec["pytest_cmd"] = cmd
        rec["pytest_stdout"] = (r.stdout or "")[-8000:]
        rec["pytest_stderr"] = (getattr(r, "stderr", "") or "")[-4000:]
        rx = await sandbox_exec(mgr, sid, "cat " + xml)
        # The raw report is preserved verbatim. It is never synthesised: if cat failed, that is
        # recorded as a missing report, not replaced with an empty document.
        rec["junit_xml"] = (rx.stdout or "") if rx.exit_code == 0 else None
        rec["junit_read_rc"] = rx.exit_code
        nodes, dupes, parse_error = _nodes_from_junit(rec["junit_xml"] or "")
        rec["nodes"] = nodes
        rec["duplicate_nodes"] = dupes
        rec["junit_parse_error"] = parse_error if rx.exit_code == 0 else "JUnit report not readable"
        rv = await sandbox_exec(mgr, sid,
            "python3 -c \"import importlib.metadata as m;"
            "print({p: m.version(p) for p in ['swegemma','adk-submission','adk-eval-core','google-adk']})\"")
        rec["environment_versions"] = (rv.stdout or "").strip()
    except BaseException as exc:
        rec["exception"] = f"{type(exc).__name__}: {exc}"
        rec["traceback"] = _tbmod.format_exc()
        rec.setdefault("nodes", None)
    finally:
        try:
            await sandbox_stop(mgr, sid)
        except BaseException as stop_exc:
            rec["teardown_error"] = f"{type(stop_exc).__name__}: {stop_exc}"
            rec["teardown_traceback"] = _tbmod.format_exc()
        # Teardown that RETURNS is not proof of teardown. Verify the arm's own sandboxes are gone,
        # with the same bounded grace the dispatch loop uses. Nothing is deleted here, and a
        # sandbox that existed before this arm is never attributed to it.
        rec["cleanup_ok"], rec["owned_sandboxes"] = confirm_cleanup(sandboxes_before)
        ACTIVE_RUN = None
    return rec


CONTROL_DIR = RESULTS / "control_evidence"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_RECHECK = []


def _persist_arm(record):
    """Write one arm's evidence the moment it exists, so a later failure cannot erase it."""
    stem = f"{record['task']}__{record['arm']}"
    raw = record.pop("junit_xml", None)
    if raw is not None:
        (CONTROL_DIR / (stem + ".junit.xml")).write_text(raw, encoding="utf-8")
    else:
        (CONTROL_DIR / (stem + ".NO_JUNIT.txt")).write_text(
            "no readable JUnit report; nothing was synthesised\n"
            + str(record.get("junit_parse_error") or ""), encoding="utf-8")
    (CONTROL_DIR / (stem + ".json")).write_text(
        json.dumps(record, indent=2, default=str), encoding="utf-8")


CONTROL_STOP_REASON = None
for _t in SELECTED:
    if CONTROL_STOP_REASON:
        break
    saved = CONTROL_EVIDENCE[_t.instance_id]
    targets = CONTROL_TARGET_NODES[_t.instance_id]
    for _arm, _phase in (("baseline", "negative"), ("reference", "positive")):
        try:
            got = run_sync(lambda _t=_t, _a=_arm: _revalidate_control(_t, _a))
        except BaseException as _exc:
            got = {"task": _t.instance_id, "arm": _arm, "nodes": None,
                   "exception": f"{type(_exc).__name__}: {_exc}",
                   "traceback": _tbmod.format_exc(),
                   "cleanup_ok": False,
                   "owned_sandboxes": ["unknown: the arm did not reach its cleanup check"]}
        want = saved[_phase]
        expected = dict(want.get("nodes") or {})
        ok_nodes, detail = compare_control_nodes(got.get("nodes"), expected, targets, _arm)
        # A sandbox this arm leaked would be snapshotted as "pre-existing" by the dispatch loop's
        # own cleanup check, so it would never be caught there. It has to be caught here.
        agree = (ok_nodes
                 and not got.get("exception")
                 and not got.get("teardown_error")
                 and got.get("cleanup_ok") is True
                 and not got.get("duplicate_nodes")
                 and got.get("pytest_exit") == want.get("pytest_exit")
                 # baseline applies no source patch, so None is correct there and 0 would
                 # be a false success; reference MUST have applied its patch.
                 and got.get("patch_rc") == (None if _arm == "baseline" else 0)
                 and got.get("test_patch_rc") == 0
                 and not [n for n, o in (got.get("nodes") or {}).items() if o == "errored"])
        got["agrees_with_saved"] = bool(agree)
        got["saved_exit"] = want.get("pytest_exit")
        got["node_comparison"] = detail
        got["n_targets"] = len(targets)
        CONTROL_RECHECK.append(got)
        _persist_arm(dict(got))          # immediately, before the next arm runs
        print(f"  {_t.instance_id:14s} {_arm:9s} exit={got.get('pytest_exit')} "
              f"(saved {want.get('pytest_exit')}) nodes={detail['n_observed']}/{detail['n_expected']} "
              f"targets={len(targets)} agrees={agree}")
        for _k in ("missing", "extra", "changed", "skipped", "targets_wrong"):
            if detail.get(_k):
                print(f"                 {_k}: {detail[_k][:4]}"
                      + (" ..." if len(detail[_k]) > 4 else ""))
        if got.get("duplicate_nodes"):
            print(f"                 DUPLICATE node identities: {got['duplicate_nodes'][:4]}")
        if got.get("junit_parse_error"):
            print(f"                 junit: {got['junit_parse_error']}")
        if got.get("exception"):
            print(f"                 EXCEPTION: {got['exception'][:150]}")
        if got.get("teardown_error"):
            print(f"                 TEARDOWN ERROR: {got['teardown_error'][:150]}")
        if not got.get("cleanup_ok"):
            print(f"                 SANDBOX NOT CLEANED UP: {got.get('owned_sandboxes')}")
        # Evidence for this arm is already on disk (_persist_arm above). Stop before any further
        # arm runs, so a leaked sandbox cannot be inherited as "pre-existing" by the next one.
        if got.get("teardown_error") or not got.get("cleanup_ok"):
            CONTROL_STOP_REASON = (
                f"{_t.instance_id}/{_arm}: control sandbox cleanup could not be confirmed"
                + (f" ({got['teardown_error']})" if got.get("teardown_error") else ""))
            print("Stopping control re-validation:", CONTROL_STOP_REASON)
            break

(RESULTS / "control_recheck.json").write_text(
    json.dumps(CONTROL_RECHECK, indent=2, default=str), encoding="utf-8")
assert not CONTROL_STOP_REASON, (
    "control sandbox cleanup failed: " + str(CONTROL_STOP_REASON) +
    "; evidence is preserved in control_evidence/, and the model is NOT started")
_expected_arms = 2 * len(SELECTED)
assert len(CONTROL_RECHECK) == _expected_arms, (
    f"only {len(CONTROL_RECHECK)} of {_expected_arms} control arms ran; do not start the model")
_bad = [f"{c['task']}/{c['arm']}" for c in CONTROL_RECHECK if not c["agrees_with_saved"]]
assert not _bad, ("controls do not reproduce under this notebook's setup: " + str(_bad) +
                  "; do not start the model")
print("ALL FOUR CONTROLS REPRODUCE UNDER THIS SETUP (both arms, full node map, targets verified)")
